In [0]:
def processLateCycleSubscriptionAdjustmentsTruUp(df_CycleData_ConsumerSubscription_Adjust,UpdatedBy="ADB_AdjustSubscription"):

    # Deduplicating data by taking only minimum cycle
    Window_Deduplicate = Window.partitionBy('CoolSculptingID','SoldToAccountID','ShipToAccountUUID','PromotionUUID','ConsumerSubscriptionUUID').orderBy(asc('CycleDate'),asc('ShipToAccountUUID'))
    
    df_FACT_ConsumerSubscription = (spark.read.table('Promotion.FACT_ConsumerSubscription')
                                    .filter("VersionId = 1")
                                    .select('ConsumerSubscriptionUUID','ShipToAccountUUID')
                                    .withColumnRenamed('ShipToAccountUUID','CS_ShipToAccountUUID')
                                    )
    
    df_CycleData_ConsumerSubscription_Adjust = (df_CycleData_ConsumerSubscription_Adjust
                                                .withColumnRenamed('ShipToAccountUUID','AD_ShipToAccountUUID')
                                                )

    df_CycleData_ConsumerSubscription_Adjust = (df_CycleData_ConsumerSubscription_Adjust
                                                .join(df_FACT_ConsumerSubscription,['ConsumerSubscriptionUUID'],'left')
                                                .withColumn('ShipToAccountUUID', when(col('CS_ShipToAccountUUID').isNull(), col('AD_ShipToAccountUUID')).otherwise(col('CS_ShipToAccountUUID')))
                                                )

    df_CycleData_AGG_CycleClassification_DeDUP = (df_CycleData_ConsumerSubscription_Adjust
                                                .withColumn('rownum',row_number().over(Window_Deduplicate))
                                                .filter('rownum = 1')
                                                .drop('rownum')
                                                )

    df_CycleData_AGG_CycleClassification_DeDUP = applyRules(df_CycleData_AGG_CycleClassification_DeDUP,'AdjustSubscription_TruUp')

    df_ConsumerSubscription_Update = (df_CycleData_AGG_CycleClassification_DeDUP
                                    .withColumn('VersionID',col('VersionID')+1)
                                    .withColumn('NewConsumerSubscriptionUUID',expr('uuid()'))
                                    .withColumn('CreatedBy',lit(UpdatedBy))
                                    .withColumn('CreatedDate',lit(current_timestamp()))
                                    .withColumn('UpdatedBy',lit(UpdatedBy))
                                    .withColumn('UpdatedDate',lit(current_timestamp()))
                                    )

    df_ConsumerSubscription_New = (df_CycleData_AGG_CycleClassification_DeDUP
                                    .withColumn('VersionID',lit(1))
                                    .withColumn('PreviousConsumerSubscriptionUUID',col('ConsumerSubscriptionUUID'))
                                    .withColumn('NewConsumerSubscriptionUUID',expr('uuid()'))
                                    .withColumn('ConsumerSubscriptionUUID',lit(None).cast(StringType()))
                                    .withColumn('CreatedBy',lit(UpdatedBy))
                                    .withColumn('CreatedDate',lit(current_timestamp()))
                                    .withColumn('UpdatedBy',lit(UpdatedBy))
                                    .withColumn('UpdatedDate',lit(current_timestamp()))
                                    )


    df_ConsumerSubscription_Final = df_ConsumerSubscription_Update.unionByName(df_ConsumerSubscription_New,allowMissingColumns=True)

    dl_FACTConsumer = DeltaTable.forName(spark, 'Promotion.FACT_ConsumerSubscription')

    (dl_FACTConsumer.alias("tgt")
            .merge(df_ConsumerSubscription_Final.alias("src"),
                ("tgt.ConsumerSubscriptionUUID = src.ConsumerSubscriptionUUID AND tgt.PromotionUUID = src.PromotionUUID"))
    .whenNotMatchedInsert(values =
    {
        "tgt.ConsumerSubscriptionUUID": "src.NewConsumerSubscriptionUUID",
        "tgt.PreviousConsumerSubscriptionUUID": "src.PreviousConsumerSubscriptionUUID",
        "tgt.CoolSculptingID": "src.CoolSculptingID",
        "tgt.ShipToAccountUUID": "src.ShipToAccountUUID",
        "tgt.SoldToAccountID": "src.SoldToAccountID",
        "tgt.PromotionUUID": "src.PromotionUUID",            
        "tgt.SubscriptionStartDate": "src.NewSubscriptionStartDate",
        "tgt.SubscriptionEndDate": "src.NewSubscriptionEndDate",
        "tgt.InvoiceExceptionFlag": "src.InvoiceExceptionFlag",
        "tgt.VersionID": "src.VersionID",
        "tgt.Comments": "src.Comments",
        "tgt.CreatedBy": "src.CreatedBy",
        "tgt.CreatedDate": "src.CreatedDate",
        "tgt.UpdatedBy": "src.UpdatedBy",
        "tgt.UpdatedDate": "src.UpdatedDate"
    }
    ).whenMatchedUpdate(set =
            {
        "tgt.VersionID": "tgt.VersionID+1",
        "tgt.Comments": "src.Comments",
        "tgt.UpdatedBy": "src.UpdatedBy",
        "tgt.UpdatedDate": "src.UpdatedDate"
            })
    .execute())



In [0]:
def processLateCycleSubscriptionReallignmentTruUp(df_CycleData_Consumer_CycleFlag,UpdatedBy="ADB_RealignSubscription"):
    df_FACT_InvoiceAddendumDetails = spark.read.table('Promotion.FACT_InvoiceAddendumDetails').filter('VersionID = 1')
    df_FACT_ConsumerSubscription = spark.read.table('Promotion.FACT_ConsumerSubscription')
    df_FACT_ConsumerSubscription_2 = spark.read.table('Promotion.FACT_ConsumerSubscription').filter('VersionID = 1')
    df_CycleData_Consumer_CycleFlag = df_CycleData_Consumer_CycleFlag.select('PromotionUUID','CoolSculptingID').drop_duplicates()

    df_InvoiceAddendumDetail_SubReallign = (df_FACT_InvoiceAddendumDetails
                                            .join(df_CycleData_Consumer_CycleFlag,['CoolSculptingID','PromotionUUID'],'inner')
                                            .join(df_FACT_ConsumerSubscription,(df_FACT_InvoiceAddendumDetails.ConsumerSubscriptionUUID == df_FACT_ConsumerSubscription.ConsumerSubscriptionUUID),'left')
                                            .join(df_FACT_ConsumerSubscription_2,
                                            (df_FACT_InvoiceAddendumDetails.CoolSculptingID == df_FACT_ConsumerSubscription_2.CoolSculptingID) 
                                            &
                                        ((df_FACT_InvoiceAddendumDetails.ShipToAccountUUID == df_FACT_ConsumerSubscription_2.ShipToAccountUUID) 
                                            |
                                         ((df_FACT_InvoiceAddendumDetails.SoldToAccountID == df_FACT_ConsumerSubscription_2.SoldToAccountID) 
                                            &
                                         (df_FACT_InvoiceAddendumDetails.ShipToAccountUUID != df_FACT_ConsumerSubscription_2.ShipToAccountUUID))
                                        )&
                                            (df_FACT_InvoiceAddendumDetails.PromotionUUID == df_FACT_ConsumerSubscription_2.PromotionUUID) 
                                            &
                                            (df_FACT_InvoiceAddendumDetails.CycleDate.between(
                                                df_FACT_ConsumerSubscription_2.SubscriptionStartDate,
                                                df_FACT_ConsumerSubscription_2.SubscriptionEndDate)),'left')
                                            .select(df_FACT_InvoiceAddendumDetails.CycleDate,
                                                    df_FACT_InvoiceAddendumDetails.ShipToAccountUUID,
                                                    df_FACT_InvoiceAddendumDetails.SoldToAccountID,
                                                    df_FACT_InvoiceAddendumDetails.PromotionUUID,
                                                    df_FACT_InvoiceAddendumDetails.CoolSculptingID,
                                                    df_FACT_InvoiceAddendumDetails.IncrementalInvoiceCharge,
                                                    df_FACT_ConsumerSubscription.ConsumerSubscriptionUUID.alias('UUID_ConsumerSubscriptionUUID'),
                                                    df_FACT_ConsumerSubscription_2.ConsumerSubscriptionUUID.alias('NKEY_ConsumerSubscriptionUUID'),
                                                    df_FACT_ConsumerSubscription.VersionID,
                                                    df_FACT_ConsumerSubscription.InvoiceExceptionFlag,
                                                    df_FACT_ConsumerSubscription.SubscriptionEndDate
                                                    )
                                            # .filter('VersionID > 1 AND NKEY_ConsumerSubscriptionUUID IS NULL')
                                            )
    
    df_InvoiceAddendumDetail_SubReallign = applyRules(df_InvoiceAddendumDetail_SubReallign,'RealignNewSubscription_TruUp')

    df_InvoiceAddendumDetail_SubReallign_NewSub = (df_InvoiceAddendumDetail_SubReallign
                                                .withColumn('ROW_NUM',
                                                            row_number().over(Window.partitionBy('ShipToAccountUUID','CoolSculptingID','PromotionUUID','RealignNewSubscriptionFlag').orderBy(asc('CycleDate'),asc('ShipToAccountUUID'))))
                                                .filter('RealignNewSubscriptionFlag = 1 AND ROW_NUM = 1')
                                                .withColumn('VersionID',lit(1))
                                                .withColumn('NewConsumerSubscriptionUUID',expr('uuid()'))
                                                .withColumn('ConsumerSubscriptionUUID',lit(None).cast(StringType()))
                                                .withColumn('CreatedBy',lit(UpdatedBy))
                                                .withColumn('CreatedDate',lit(current_timestamp()))
                                                .withColumn('UpdatedBy',lit(UpdatedBy))
                                                .withColumn('UpdatedDate',lit(current_timestamp()))
                                            )
    
    dl_FACTConsumer = DeltaTable.forName(spark, 'Promotion.FACT_ConsumerSubscription')

    (dl_FACTConsumer.alias("tgt")
        .merge(df_InvoiceAddendumDetail_SubReallign_NewSub.alias("src"),
            ("tgt.CoolSculptingID = src.CoolSculptingID AND tgt.ShipToAccountUUID = src.ShipToAccountUUID AND tgt.PromotionUUID = src.PromotionUUID AND tgt.SubscriptionStartDate = src.NewSubscriptionStartDate AND tgt.VersionID = src.VersionID"))   
    .whenNotMatchedInsert(values =
    {
    "tgt.ConsumerSubscriptionUUID": "src.NewConsumerSubscriptionUUID",
    "tgt.CoolSculptingID": "src.CoolSculptingID",
    "tgt.ShipToAccountUUID": "src.ShipToAccountUUID",
    "tgt.SoldToAccountID": "src.SoldToAccountID",    
    "tgt.PromotionUUID": "src.PromotionUUID",            
    "tgt.SubscriptionStartDate": "src.NewSubscriptionStartDate",
    "tgt.SubscriptionEndDate": "src.NewSubscriptionEndDate",
    "tgt.InvoiceExceptionFlag": "src.InvoiceExceptionFlag",    
    "tgt.VersionID": "src.VersionID",
    "tgt.Comments": "src.Comments",
    "tgt.CreatedBy": "src.CreatedBy",
    "tgt.CreatedDate": "src.CreatedDate",
    "tgt.UpdatedBy": "src.UpdatedBy",
    "tgt.UpdatedDate": "src.UpdatedDate"
    }
    )
    .execute())


In [0]:
def processLateCycleSubscriptionAdjustmentsTruUp_ConsumerBased(df_CycleData_ConsumerSubscription_Adjust,UpdatedBy="ADB_AdjustSubscription"):

    # Deduplicating data by taking only minimum cycle
    Window_Deduplicate = Window.partitionBy('CoolSculptingID','PromotionUUID','ConsumerSubscriptionUUID').orderBy(asc('CycleDate'))
    
    df_FACT_ConsumerSubscription = (spark.read.table('Promotion.FACT_ConsumerSubscription')
                                    .filter("VersionId = 1")
                                    .select('ConsumerSubscriptionUUID','ShipToAccountUUID')
                                    .withColumnRenamed('ShipToAccountUUID','CS_ShipToAccountUUID')
                                    )
    
    df_CycleData_ConsumerSubscription_Adjust = (df_CycleData_ConsumerSubscription_Adjust
                                                .withColumnRenamed('ShipToAccountUUID','AD_ShipToAccountUUID')
                                                )

    df_CycleData_ConsumerSubscription_Adjust = (df_CycleData_ConsumerSubscription_Adjust
                                                .join(df_FACT_ConsumerSubscription,['ConsumerSubscriptionUUID'],'left')
                                                .withColumn('ShipToAccountUUID', when(col('CS_ShipToAccountUUID').isNull(), col('AD_ShipToAccountUUID')).otherwise(col('CS_ShipToAccountUUID')))
                                                )

    df_CycleData_AGG_CycleClassification_DeDUP = (df_CycleData_ConsumerSubscription_Adjust
                                                .withColumn('rownum',row_number().over(Window_Deduplicate))
                                                .filter('rownum = 1')
                                                .drop('rownum')
                                                )

    df_CycleData_AGG_CycleClassification_DeDUP = applyRules(df_CycleData_AGG_CycleClassification_DeDUP,'AdjustSubscription_TruUp')

    df_ConsumerSubscription_Update = (df_CycleData_AGG_CycleClassification_DeDUP
                                    .withColumn('VersionID',col('VersionID')+1)
                                    .withColumn('NewConsumerSubscriptionUUID',expr('uuid()'))
                                    .withColumn('CreatedBy',lit(UpdatedBy))
                                    .withColumn('CreatedDate',lit(current_timestamp()))
                                    .withColumn('UpdatedBy',lit(UpdatedBy))
                                    .withColumn('UpdatedDate',lit(current_timestamp()))
                                    )

    df_ConsumerSubscription_New = (df_CycleData_AGG_CycleClassification_DeDUP
                                    .withColumn('VersionID',lit(1))
                                    .withColumn('PreviousConsumerSubscriptionUUID',col('ConsumerSubscriptionUUID'))
                                    .withColumn('NewConsumerSubscriptionUUID',expr('uuid()'))
                                    .withColumn('ConsumerSubscriptionUUID',lit(None).cast(StringType()))
                                    .withColumn('CreatedBy',lit(UpdatedBy))
                                    .withColumn('CreatedDate',lit(current_timestamp()))
                                    .withColumn('UpdatedBy',lit(UpdatedBy))
                                    .withColumn('UpdatedDate',lit(current_timestamp()))
                                    )


    df_ConsumerSubscription_Final = df_ConsumerSubscription_Update.unionByName(df_ConsumerSubscription_New,allowMissingColumns=True)

    dl_FACTConsumer = DeltaTable.forName(spark, 'Promotion.FACT_ConsumerSubscription')

    (dl_FACTConsumer.alias("tgt")
            .merge(df_ConsumerSubscription_Final.alias("src"),
                ("tgt.ConsumerSubscriptionUUID = src.ConsumerSubscriptionUUID AND tgt.PromotionUUID = src.PromotionUUID"))
    .whenNotMatchedInsert(values =
    {
        "tgt.ConsumerSubscriptionUUID": "src.NewConsumerSubscriptionUUID",
        "tgt.PreviousConsumerSubscriptionUUID": "src.PreviousConsumerSubscriptionUUID",
        "tgt.CoolSculptingID": "src.CoolSculptingID",
        "tgt.ShipToAccountUUID": "src.ShipToAccountUUID",
        "tgt.SoldToAccountID": "src.SoldToAccountID",
        "tgt.PromotionUUID": "src.PromotionUUID",            
        "tgt.SubscriptionStartDate": "src.NewSubscriptionStartDate",
        "tgt.SubscriptionEndDate": "src.NewSubscriptionEndDate",
        "tgt.InvoiceExceptionFlag": "src.InvoiceExceptionFlag",
        "tgt.VersionID": "src.VersionID",
        "tgt.Comments": "src.Comments",
        "tgt.CreatedBy": "src.CreatedBy",
        "tgt.CreatedDate": "src.CreatedDate",
        "tgt.UpdatedBy": "src.UpdatedBy",
        "tgt.UpdatedDate": "src.UpdatedDate"
    }
    ).whenMatchedUpdate(set =
            {
        "tgt.VersionID": "tgt.VersionID+1",
        "tgt.Comments": "src.Comments",
        "tgt.UpdatedBy": "src.UpdatedBy",
        "tgt.UpdatedDate": "src.UpdatedDate"
            })
    .execute())



In [0]:
def processLateCycleSubscriptionReallignmentTruUp_ConsumerBased(df_CycleData_Consumer_CycleFlag,UpdatedBy="ADB_RealignSubscription"):
    df_FACT_InvoiceAddendumDetails = spark.read.table('Promotion.FACT_InvoiceAddendumDetails').filter('VersionID = 1')
    df_FACT_ConsumerSubscription = spark.read.table('Promotion.FACT_ConsumerSubscription')
    df_FACT_ConsumerSubscription_2 = spark.read.table('Promotion.FACT_ConsumerSubscription').filter('VersionID = 1')
    df_CycleData_Consumer_CycleFlag = df_CycleData_Consumer_CycleFlag.select('PromotionUUID','CoolSculptingID').drop_duplicates()

    df_InvoiceAddendumDetail_SubReallign = (df_FACT_InvoiceAddendumDetails
                                            .join(df_CycleData_Consumer_CycleFlag,['CoolSculptingID','PromotionUUID'],'inner')
                                            .join(df_FACT_ConsumerSubscription,(df_FACT_InvoiceAddendumDetails.ConsumerSubscriptionUUID == df_FACT_ConsumerSubscription.ConsumerSubscriptionUUID),'left')
                                            .join(df_FACT_ConsumerSubscription_2,
                                            (df_FACT_InvoiceAddendumDetails.CoolSculptingID == df_FACT_ConsumerSubscription_2.CoolSculptingID) 
                                            &
                                            (df_FACT_InvoiceAddendumDetails.PromotionUUID == df_FACT_ConsumerSubscription_2.PromotionUUID) 
                                            &
                                            (df_FACT_InvoiceAddendumDetails.CycleDate.between(
                                                df_FACT_ConsumerSubscription_2.SubscriptionStartDate,
                                                df_FACT_ConsumerSubscription_2.SubscriptionEndDate)),'left')
                                            .select(df_FACT_InvoiceAddendumDetails.CycleDate,
                                                    df_FACT_InvoiceAddendumDetails.ShipToAccountUUID,
                                                    df_FACT_InvoiceAddendumDetails.SoldToAccountID,
                                                    df_FACT_InvoiceAddendumDetails.PromotionUUID,
                                                    df_FACT_InvoiceAddendumDetails.CoolSculptingID,
                                                    df_FACT_InvoiceAddendumDetails.IncrementalInvoiceCharge,
                                                    df_FACT_ConsumerSubscription.ConsumerSubscriptionUUID.alias('UUID_ConsumerSubscriptionUUID'),
                                                    df_FACT_ConsumerSubscription_2.ConsumerSubscriptionUUID.alias('NKEY_ConsumerSubscriptionUUID'),
                                                    df_FACT_ConsumerSubscription.VersionID,
                                                    df_FACT_ConsumerSubscription.InvoiceExceptionFlag,
                                                    df_FACT_ConsumerSubscription.SubscriptionEndDate
                                                    )
                                            # .filter('VersionID > 1 AND NKEY_ConsumerSubscriptionUUID IS NULL')
                                            )
    
    df_InvoiceAddendumDetail_SubReallign = applyRules(df_InvoiceAddendumDetail_SubReallign,'RealignNewSubscription_TruUp')

    df_InvoiceAddendumDetail_SubReallign_NewSub = (df_InvoiceAddendumDetail_SubReallign
                                                .withColumn('ROW_NUM',
                                                            row_number().over(Window.partitionBy('CoolSculptingID','PromotionUUID','RealignNewSubscriptionFlag').orderBy(asc('CycleDate'))))
                                                .filter('RealignNewSubscriptionFlag = 1 AND ROW_NUM = 1')
                                                .withColumn('VersionID',lit(1))
                                                .withColumn('NewConsumerSubscriptionUUID',expr('uuid()'))
                                                .withColumn('ConsumerSubscriptionUUID',lit(None).cast(StringType()))
                                                .withColumn('CreatedBy',lit(UpdatedBy))
                                                .withColumn('CreatedDate',lit(current_timestamp()))
                                                .withColumn('UpdatedBy',lit(UpdatedBy))
                                                .withColumn('UpdatedDate',lit(current_timestamp()))
                                            )
    
    dl_FACTConsumer = DeltaTable.forName(spark, 'Promotion.FACT_ConsumerSubscription')
    
    (dl_FACTConsumer.alias("tgt")
        .merge(df_InvoiceAddendumDetail_SubReallign_NewSub.alias("src"),
            ("tgt.CoolSculptingID = src.CoolSculptingID AND tgt.PromotionUUID = src.PromotionUUID AND tgt.SubscriptionStartDate = src.NewSubscriptionStartDate AND tgt.VersionID = src.VersionID"))   
    .whenNotMatchedInsert(values =
    {
    "tgt.ConsumerSubscriptionUUID": "src.NewConsumerSubscriptionUUID",
    "tgt.CoolSculptingID": "src.CoolSculptingID",
    "tgt.ShipToAccountUUID": "src.ShipToAccountUUID",
    "tgt.SoldToAccountID": "src.SoldToAccountID",    
    "tgt.PromotionUUID": "src.PromotionUUID",            
    "tgt.SubscriptionStartDate": "src.NewSubscriptionStartDate",
    "tgt.SubscriptionEndDate": "src.NewSubscriptionEndDate",
    "tgt.InvoiceExceptionFlag": "src.InvoiceExceptionFlag",    
    "tgt.VersionID": "src.VersionID",
    "tgt.Comments": "src.Comments",
    "tgt.CreatedBy": "src.CreatedBy",
    "tgt.CreatedDate": "src.CreatedDate",
    "tgt.UpdatedBy": "src.UpdatedBy",
    "tgt.UpdatedDate": "src.UpdatedDate"
    }
    )
    .execute())
